In [2]:
import os 
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import pickle
import anndata
from anndata import AnnData
from typing import Dict, Optional, Any
import json

In [3]:
def pickle_data(data, path):

    with open(path, "wb") as f:
        pickle.dump(data, f)

def load_pickle(path): 

    with open(path, "rb") as f: 
        return pickle.load(f)

### Analyze dataset properties

In [4]:
datasets = ["TCGA", "Hartwig", "CPTAC", "TARGET_AML"]

In [5]:
def get_feature_value_counts(embeddings, feature):
    all_values = []
    for patient_id, embedding in embeddings.items():
        val = embedding.get(feature, None)
        if val not in [None, np.nan, "Unknown", "unknown", "'--", "NA", "na", "nan"]:
            all_values.append(val)
    return pd.Series(all_values).value_counts()

In [6]:
vals = []

for data in datasets:
    print(f"Dataset: {data}")
    
    # Get number of genes
    genes_path = f"../data/{data}/{data}_genes_ordered_by_variance.pkl"
    genes = load_pickle(genes_path)
    num_genes = len(genes)
    
    # Look at the embeddings
    embeddings_path = f"../data/RNA_embeddings/{data}_processed_cleaned.pkl" if data == "TCGA" else f"../data/RNA_embeddings/{data}_processed.pkl"
    embeddings = load_pickle(embeddings_path)
    
    num_cancer_types = len(get_feature_value_counts(embeddings, "cancer_type"))
    num_organs = len(get_feature_value_counts(embeddings, "organ"))
    
    if data in ["TCGA", "CPTAC"]:
        num_patients = len(embeddings)
    elif data == "TARGET_AML":
        num_patients = len(get_feature_value_counts(embeddings, "patient_id"))
    else:  # Hartwig
        mapping_file = "../data/Hartwig/sample_to_patient_mapping.pkl"
        sample_to_patient_mapping = load_pickle(mapping_file)
        Hartwig_samples = embeddings.keys()
        Hartwig_patients = set()
        for sample in Hartwig_samples:
            if sample in sample_to_patient_mapping:
                Hartwig_patients.add(sample_to_patient_mapping[sample])
        num_patients = len(Hartwig_patients)
    
    num_samples = num_patients if data in ["TCGA", "CPTAC"] else len(embeddings)
    
    # Compute PFS stats
    pfs_times  = [v["PFS_time"]  for v in embeddings.values() if v.get("PFS_time")  is not None]
    pfs_events = [v["PFS_event"] for v in embeddings.values() if v.get("PFS_event") is not None]
    median_pfs    = float(np.median(pfs_times)) if pfs_times else float("nan")
    event_rate    = float(np.mean([bool(e) for e in pfs_events])) * 100 if pfs_events else float("nan")

    vals.append([data, num_patients, num_samples, num_genes, num_cancer_types, num_organs,
                 f"{median_pfs:.1f}", f"{event_rate:.1f}%"])

df = pd.DataFrame(
    vals,
    columns=["Dataset", "# Patients", "# Samples", "# Genes",
             "# Cancer Types", "# Primary Organs", "Median PFS (days)", "PFS Event Rate"]
)
df.to_csv("../data/dataset_summary.csv", index=False)
print(df)

Dataset: TCGA
Dataset: Hartwig
Dataset: CPTAC
Dataset: TARGET_AML
      Dataset  # Patients  # Samples  # Genes  # Cancer Types  \
0        TCGA       10388      10388    17927              33   
1     Hartwig        2153       2232    18005              57   
2       CPTAC        1222       1222    18775              16   
3  TARGET_AML        1902       2586    16671               1   

   # Primary Organs Median PFS (days) PFS Event Rate  
0                26             579.0          42.4%  
1                45             154.0          84.6%  
2                16             584.5          45.9%  
3                 1             483.0          61.8%  


In [16]:
cptac_embeddings = load_pickle("../data/RNA_embeddings/CPTAC_processed.pkl")
old_cptac_embeddings = load_pickle("../data/RNA_embeddings/old_CPTAC_processed.pkl")

In [17]:
PFS_time_mismatch_count = 0
PFS_event_mismatch_count = 0

for pid in cptac_embeddings.keys():
    if pid in old_cptac_embeddings:
        new_pfs_time = cptac_embeddings[pid].get("PFS_time", None)
        old_pfs_time = old_cptac_embeddings[pid].get("PFS_time", None)
        if new_pfs_time != old_pfs_time:
            PFS_time_mismatch_count += 1
        
        new_pfs_event = cptac_embeddings[pid].get("PFS_event", None)
        old_pfs_event = old_cptac_embeddings[pid].get("PFS_event", None)
        if new_pfs_event != old_pfs_event:
            PFS_event_mismatch_count += 1

In [19]:
def compute_pfs_stats(embeddings):
    pfs_times  = [v["PFS_time"]  for v in embeddings.values() if v.get("PFS_time")  is not None]
    pfs_events = [v["PFS_event"] for v in embeddings.values() if v.get("PFS_event") is not None]
    median_pfs    = float(np.median(pfs_times)) if pfs_times else float("nan")
    event_rate    = float(np.mean([bool(e) for e in pfs_events])) * 100 if pfs_events else float("nan")
    
    return median_pfs, event_rate

In [18]:
print(f"PFS_time mismatches: {PFS_time_mismatch_count}")
print(f"PFS_event mismatches: {PFS_event_mismatch_count}")

PFS_time mismatches: 132
PFS_event mismatches: 251


In [20]:
compute_pfs_stats(cptac_embeddings)

(584.5, 45.90834697217676)

In [21]:
compute_pfs_stats(old_cptac_embeddings)

(582.5, 25.368248772504092)

### Analyze Embeddings

In [4]:
# Examine number of saved foundation model embeddings for each dataset and for each foundation model.
FMs = ["BulkRNABert", "BulkFormer", "COMPASS", "scGPT", "STATE", "CancerFoundation"]
embedding_size_dict = {
    "BulkRNABert": 256,
    "BulkFormer": 640,
    "COMPASS": 44,
    "scGPT": 512,
    "STATE": 2058,
    "CancerFoundation": 256
}

vals = []
for data in datasets:
    row = []
    for fm in FMs:
        fm_dir= f"../data/{data}/RNA/FM_embeddings/{fm}"
        
        # each sample has a pickle file stored in the fm_path, count number of files in the fm_path
        
        if os.path.exists(fm_dir):
            # num files should only include .pkl files
            num_files = len([f for f in os.listdir(fm_dir) if f.endswith(".pkl")])
            if num_files > 0:
                example_embedding = load_pickle(os.path.join(fm_dir, os.listdir(fm_dir)[0]))
                assert example_embedding.size == embedding_size_dict[fm] or example_embedding.shape[1] == embedding_size_dict[fm], f"Embedding size for {fm} is not correct. Expected {embedding_size_dict[fm]}, got {example_embedding.shape[0]}"
        else:
            num_files = 0
        
        row.append(num_files)
        
    vals.append([data] + row)
fm_df = pd.DataFrame(vals, columns=["Dataset"] + FMs)
fm_df

,Dataset,BulkRNABert,BulkFormer,COMPASS,scGPT,STATE,CancerFoundation
0,TCGA,10478,10478,10517,10478,10478,10478
1,Hartwig,2644,2644,2113,2644,2644,2644
2,CPTAC,2393,2393,2100,2393,2393,2393
3,ICGC,105,105,96,105,105,105
4,CALGB,389,389,389,389,389,389
5,TARGET_AML,2634,2634,2634,2634,2634,2634


In [7]:
vals = []

for data in datasets:
    ssgsea_path = f"../data/{data}/ssGSEA_scores/"
    pca_50_path = f"../data/{data}/PCA/PCA_50/"
    pca_100_path = f"../data/{data}/PCA/PCA_100/"
    
    num_ssgsea = len(os.listdir(ssgsea_path)) if os.path.exists(ssgsea_path) else 0
    num_pca_50 = len(os.listdir(pca_50_path)) if os.path.exists(pca_50_path) else 0
    num_pca_100 = len(os.listdir(pca_100_path)) if os.path.exists(pca_100_path) else 0
    
    example_pca_50_embedding = load_pickle(os.path.join(pca_50_path, os.listdir(pca_50_path)[0])) if num_pca_50 > 0 else None
    assert example_pca_50_embedding is None or example_pca_50_embedding.size == 50, f"PCA 50 embedding size is not correct. Expected 50, got {example_pca_50_embedding.shape[1]}"
    example_pca_100_embedding = load_pickle(os.path.join(pca_100_path, os.listdir(pca_100_path)[0])) if num_pca_100 > 0 else None
    assert example_pca_100_embedding is None or example_pca_100_embedding.size == 100, f"PCA 100 embedding size is not correct. Expected 100, got {example_pca_100_embedding.shape[1]}"
    
    vals.append([data, num_ssgsea, num_pca_50, num_pca_100])

feature_df = pd.DataFrame(vals, columns=["Dataset", "ssGSEA", "PCA_50", "PCA_100"])
feature_df

,Dataset,ssGSEA,PCA_50,PCA_100
0,TCGA,4,10478,10478
1,Hartwig,4,2644,2644
2,CPTAC,4,2393,2393
3,ICGC,4,105,105
4,CALGB,4,389,389
5,TARGET_AML,4,2634,2634


In [6]:
for data in datasets:
    embeddings_path = f"../data/RNA_embeddings/{data}_processed.pkl"
    print(f"{data} embeddings file exists: {os.path.exists(embeddings_path)}")

TCGA embeddings file exists: True
Hartwig embeddings file exists: True
CPTAC embeddings file exists: True
ICGC embeddings file exists: False
CALGB embeddings file exists: True
TARGET_AML embeddings file exists: False


### Analyze runs

In [6]:
import os
import pandas as pd
from itertools import product
from collections import Counter

run_dir = "../model_outputs/TCGA/survival/PFS"

FM_BULK  = ["BulkFormer", "BulkRNABert", "COMPASS"]
FM_SC    = ["scGPT", "STATE", "CancerFoundation"]
HVG      = [("RNA_seq", n) for n in [18, 50, 132, 500, 729, 1250, 5000, 10000]]
GS       = [(f"{agg}_{gs}", None) for agg in ["ssGSEA", "max"]
                                  for gs in ["AURORA", "immuno", "hallmark", "combined"]]
BASELINE = [("PCA_50", None), ("PCA_100", None), ("RNA_seq", "None")]
FM       = [(k, None) for k in FM_BULK + FM_SC]

ALL_EMBEDDINGS = HVG + GS + BASELINE + FM
ARCHS  = ["ResidualMLP", "Linear"]
LOSSES = ["Cox", "CIndex"]
SPLITS = [0, 1, 2, 3, 4]
SEEDS  = [37, 42]

existing = list(os.listdir(run_dir)) if os.path.exists(run_dir) else []
missing, found = [], []
dup_rows = []

for (key, nhvg), arch, loss, split, seed in product(ALL_EMBEDDINGS, ARCHS, LOSSES, SPLITS, SEEDS):
    prefix = f"{arch}_{loss}_rnaKey_{key}_nHVGs_{nhvg}_seed_{seed}_split_{split}_"
    matches = [d for d in existing if d.startswith(prefix)]
    if matches:
        found.append(prefix)
        if len(matches) > 1:
            dup_rows.append({"key": key, "nHVGs": nhvg, "arch": arch, "loss": loss,
                             "split": split, "seed": seed, "n_duplicates": len(matches)})
    else:
        missing.append(prefix)

print(f"Expected: {len(found)+len(missing)} | Found: {len(found)} | Missing: {len(missing)}")

dup_df = pd.DataFrame(dup_rows)
print(f"\nDuplicated runs: {len(dup_df)}")
display(dup_df)

Expected: 1000 | Found: 1000 | Missing: 0

Duplicated runs: 0


""


In [9]:
target_embeddings = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/RNA_embeddings/TARGET_AML_processed.pkl")

In [10]:
list(target_embeddings.keys())[:5]

['TARGET-20-PASNIY-09A',
 'TARGET-20-PASNKC-09A',
 'TARGET-20-PASNLR-03A',
 'TARGET-20-PASNAL-09A',
 'TARGET-20-PASNDA-09A']

In [13]:
target_embeddings['TARGET-20-PASNIY-09A']['PFS_time']

1962.0

In [5]:
import pandas as pd
from itertools import product

CSV_PATH = "/data/rbg/users/ashyamal/RNA-FM-benchmarking/model_outputs/TCGA/survival/PFS/all_metrics.csv"

ARCHS = ["ResidualMLP", "Linear"]
LOSSES = ["Cox", "CIndex"]
SPLITS = [0, 1, 2, 3, 4]
SEEDS = [37, 42]

FM_KEYS = ["STATE", "BulkRNABert", "BulkFormer", "COMPASS", "scGPT", "CancerFoundation"]

GS_KEYS = [
    "max_combined", "max_AURORA", "max_hallmark", "max_immuno",
    "ssGSEA_combined", "ssGSEA_AURORA", "ssGSEA_hallmark", "ssGSEA_immuno",
]

RNA_SEQ_HVGS = [18, 50, 132, 500, 729, 1250, 5000, 10000]

def normalize_nhvg(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in {"", "none", "nan"}:
        return None
    try:
        return int(float(x))
    except Exception:
        return x

def main():
    df = pd.read_csv(CSV_PATH)

    df = df[["dataset", "task", "arch", "loss", "RNA_key", "nHVGs", "seed", "split"]].copy()
    df["nHVGs"] = df["nHVGs"].apply(normalize_nhvg)

    expected_settings = []

    # RNA_seq HVG settings
    for n in RNA_SEQ_HVGS:
        expected_settings.append(("RNA_seq", n))

    # RNA_seq no-dim-red baseline
    expected_settings.append(("RNA_seq", None))

    # gene-set settings
    for k in GS_KEYS:
        expected_settings.append((k, None))

    # PCA baselines
    expected_settings.extend([
        ("PCA_50", None),
        ("PCA_100", None),
    ])

    # FM settings
    for k in FM_KEYS:
        expected_settings.append((k, None))

    assert len(expected_settings) == 25, f"Expected 25 settings, got {len(expected_settings)}"

    expected_rows = []
    for (rna_key, nhvg), arch, loss, split, seed in product(
        expected_settings, ARCHS, LOSSES, SPLITS, SEEDS
    ):
        expected_rows.append({
            "dataset": "TCGA",
            "task": "survival",
            "arch": arch,
            "loss": loss,
            "RNA_key": rna_key,
            "nHVGs": nhvg,
            "seed": seed,
            "split": split,
        })

    expected_df = pd.DataFrame(expected_rows)
    existing_df = df.drop_duplicates()

    merged = expected_df.merge(
        existing_df,
        on=["dataset", "task", "arch", "loss", "RNA_key", "nHVGs", "seed", "split"],
        how="left",
        indicator=True,
    )

    missing_df = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

    print(f"Expected rows: {len(expected_df)}")
    print(f"Existing unique rows: {len(existing_df)}")
    print(f"Missing rows: {len(missing_df)}")
    print(f"Matched rows: {len(expected_df) - len(missing_df)}")
    print()

    if len(missing_df) == 0:
        print("No missing rows.")
    else:
        print(missing_df.sort_values(
            ["RNA_key", "nHVGs", "arch", "loss", "seed", "split"]
        ).to_string(index=False))

if __name__ == "__main__":
    main()

Expected rows: 1000
Existing unique rows: 992
Missing rows: 8
Matched rows: 992

dataset     task        arch   loss          RNA_key  nHVGs  seed  split
   TCGA survival ResidualMLP CIndex       BulkFormer    NaN    37      4
   TCGA survival ResidualMLP    Cox       BulkFormer    NaN    37      4
   TCGA survival      Linear CIndex      BulkRNABert    NaN    37      0
   TCGA survival      Linear    Cox      BulkRNABert    NaN    37      0
   TCGA survival ResidualMLP CIndex CancerFoundation    NaN    37      0
   TCGA survival ResidualMLP    Cox CancerFoundation    NaN    37      0
   TCGA survival      Linear CIndex            scGPT    NaN    37      4
   TCGA survival      Linear    Cox            scGPT    NaN    37      3
